# Vendor Performance Analytics - Exploratory Data Analysis (EDA)
**Author:** Data Analytics Portfolio Project  
**Pipeline:** Excel -> Pandas -> PostgreSQL -> Power BI Interactive Analytics  

This notebook walks through the data profiling, cleaning decisions, margin attribution, price variance detection, and the multi-dimensional Vendor Performance Scoring (VPS) algorithm.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to sys.path
BASE_DIR = os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd()
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from src.ingestion.loader import load_raw_data
from src.cleaning.cleaner import clean_all_datasets
from src.transformation.metrics import calculate_all_metrics
from src.transformation.scoring import calculate_vendor_scores

print('Imports initialized successfully!')

## 1. Raw Data Ingestion & Data Quality Profiling
We inspect the multi-table relational dataset: `vendors`, `products`, `purchases`, `sales`, and `inventory`.

In [ ]:
raw_data = load_raw_data(os.path.join(BASE_DIR, 'data', 'raw'))
for name, df in raw_data.items():
    print(f'Table: {name:<12} | Rows: {len(df):>6} | Columns: {len(df.columns)}')
    
raw_purchases = raw_data['purchases']
print('\nRaw Purchases Sample with Noise (duplicates & negative quantities):')
display(raw_purchases.head(5))

## 2. Data Cleaning & Validation Execution
We execute the standardized cleaning pipeline: schema normalization, date parsing, outlier removal via IQR benchmarks, and duplicate removal.

In [ ]:
cleaned_data = clean_all_datasets(raw_data, os.path.join(BASE_DIR, 'data', 'processed'))
print(f'Valid Clean Purchases: {len(cleaned_data["purchases"])} records')
print(f'Valid Clean Sales:     {len(cleaned_data["sales"])} records')

## 3. Metrics Engineering & Vendor Scoring Methodology
Compute procurement spend, attributed sales, gross margin %, purchase-to-sales ratio, and weighted VPS scores.

In [ ]:
metrics = calculate_all_metrics(cleaned_data, os.path.join(BASE_DIR, 'data', 'processed'))
scores = calculate_vendor_scores(metrics['vendor_summary'], os.path.join(BASE_DIR, 'data', 'processed'))

# Display Top 10 Ranked Vendors
cols = ['vendor_id', 'vendor_name', 'vendor_category', 'total_procurement_cost', 'total_revenue', 'gross_margin_pct', 'vendor_performance_score', 'performance_tier']
display(scores[cols].head(10))

## 4. Key Strategic Insights & Profitability Frontier
Analyze vendor gross margins and identify suppliers exhibiting pricing anomalies.

In [ ]:
print('Strategic Highlights:')
top_vps = scores.iloc[0]
print(f'1. Highest Rated Supplier: {top_vps["vendor_name"]} (VPS: {top_vps["vendor_performance_score"]})')
print(f'   Tier: {top_vps["performance_tier"]}')
print(f'   Strategic Action: {top_vps["recommendation"]}')

lowest_margin = scores.sort_values('gross_margin_pct').iloc[0]
print(f'\n2. Compressed Margin Alert: {lowest_margin["vendor_name"]} (Margin: {lowest_margin["gross_margin_pct"]}%)')
print(f'   Purchase-to-Sales Ratio: {lowest_margin["purchase_to_sales_ratio"]}')